##### Single Ingest

In [3]:
pip install pyroaring

Note: you may need to restart the kernel to use updated packages.


In [4]:
from pyroaring import BitMap

In [5]:
from datetime import datetime

class SearchEngine:
    def __init__(self, ignored_fields=None):
        self.schema = {}
        self.index = {}
        self.documents = {}
        self.next_doc_id = 0
        self.ignored_fields = {
            field.lower()
            for field in (ignored_fields or set())
        }

    # Determine field type
    def determine_type(self, value):
        if isinstance(value, bool):
            return "bool"
        elif isinstance(value, int):
            return "int"
        elif isinstance(value, float):
            return "float"
        elif isinstance(value, str):
            try:
                datetime.strptime(value, "%Y-%m-%d")
                return "date"
            except ValueError:
                return "categorical"
        return "unknown"

    # Build schema
    def build_schema(self, doc):
        for key, value in doc.items():
            field_type = self.determine_type(value)
            self.schema[key] = {
                "type": field_type,
                "indexed": (
                    field_type == "categorical"
                    and key.lower() not in self.ignored_fields
                )
            }

    # Index one document
    def index_document(self, doc):
        # Build schema only once
        if not self.schema:
            self.build_schema(doc)
        doc_id = self.next_doc_id
        # Save original document
        self.documents[doc_id] = doc
        # Update bitmap index
        for field, value in doc.items():
            # Ignore fields that aren't indexed
            if not self.schema[field]["indexed"]:
                continue
            if field not in self.index:
                self.index[field] = {}
            if value not in self.index[field]:
                self.index[field][value] = BitMap()
            self.index[field][value].add(doc_id)
        self.next_doc_id += 1

    # Equality lookup
    def lookup(self, field, value):
        if field not in self.index:
            return BitMap()
        if value not in self.index[field]:
            return BitMap()
        return self.index[field][value]

    def query_and(self, results):
        if not results:
            return BitMap()
        answer = results[0].copy()
        for result in results[1:]:
            answer &= result
        return answer

    def query_or(self, results):
        if not results:
            return BitMap()
        answer = results[0].copy()
        for result in results[1:]:
            answer |= result
        return answer

    def query_not(self, result):
        all_docs = BitMap(self.documents.keys())
        return all_docs - result

    def query_in(self, field, values):
        results = []
        for value in values:
            results.append(self.lookup(field, value))
        return self.query_or(results)

    def query_not_in(self, field, values):
        result = self.query_in(field, values)
        return self.query_not(result)

    # batch ingest
    def index_documents(self, docs):

        for doc in docs:
            self.index_document(doc)

    #cardinality Analysis

    def analyse_cardinality(self):
        for field in self.index:
            self.schema[field]["cardinality"] = len(self.index[field])
        return self.schema

    # def disable_high_cardinality(self,threshold):
    #     self.analyse_cardinality()
    #     for field in self.schema:
    #         if self.schema[field]["cardinality"]>threshold:
    #             self.schema[field]["indexed"]=False

In [6]:
doc1 = {
    "name": "Alex",
    "theme": "dark",
    "country": "USA"
}

doc2 = {
    "name": "John",
    "theme": "dark",
    "country": "India"
}

doc3 = {
    "name": "Alex",
    "theme": "light",
    "country": "India"
}


engine = SearchEngine()

engine.index_document(doc1)
engine.index_document(doc2)
engine.index_document(doc3)

In [7]:
# doc4 = {
#     "name": "Pragya",
#     "theme": "light",
#     "country": "India"
# }
# engine.index_document(doc4)

In [8]:
# print(engine.get_cardinality())

In [9]:
# engine.disable_high_cardinality(2)

In [10]:
# doc5 = {
#     "name": "Rachna",
#     "theme": "light",
#     "country": "India"
# }
# engine.index_document(doc5)

In [11]:
#name not updated becoz name cardinality false
# print(engine.get_cardinality())

In [12]:
engine.lookup("name", "Alex")
# {0,2}

BitMap([0, 2])

In [13]:
engine.query_and([
    engine.lookup("name","Alex"),
    engine.lookup("theme","dark")
])
# {0}

BitMap([0])

In [14]:
engine.query_or([
    engine.lookup("name","Alex"),
    engine.lookup("country","India")
])

BitMap([0, 1, 2])

In [15]:
engine.query_not(
    engine.lookup("country","USA")
)
# {1,2}

BitMap([1, 2])

In [16]:
engine.query_in(
    "country",
    ["USA","India"]
)
# {0,1,2}

BitMap([0, 1, 2])

In [17]:
engine.query_not_in(
    "theme",
    ["dark"]
)
# {2}

BitMap([2])

### BSI

In [18]:
docs = [
    {"id": 0, "age": 5},
    {"id": 1, "age": 3},
    {"id": 2, "age": 6},
    {"id": 3, "age": 1},
    {"id": 4, "age": 7},
    {"id": 5, "age": 4},
    {"id": 6, "age": 2},
    {"id": 7, "age": 0}
]

| Doc | Age | Binary |
| --: | --: | :----: |
|   0 |   5 |   101  |
|   1 |   3 |   011  |
|   2 |   6 |   110  |
|   3 |   1 |   001  |
|   4 |   7 |   111  |
|   5 |   4 |   100  |
|   6 |   2 |   010  |
|   7 |   0 |   000  |


Look at the leftmost bit.Which document IDs have bit 2 = 1? 

0 2 4 5

similarly, Slice1 = BitMap([1,2,4,6])

and Slice0 = BitMap([0,1,3,4])

In [ ]:
class BSI:
  def __init__(self):
    self.slices=[]
    self.all_docs=BitMap()

  def ensure_slices(self,value):
    while len(self.slices)<value.bit_length():
      self.slices.append(BitMap())

  def add(self,doc_id,value):
    self.ensure_slices(value)
    self.all_docs.add(doc_id)
    for i in range(value.bit_length()):
      if ((value>>i)&1)==1:       #if the i'th bit is 1 we add the document to slice[i]
        self.slices[i].add(doc_id)

  def print_slices(self):
    for i, bitmap in enumerate(self.slices):
        print(f"Slice {i}: {list(bitmap)}")

  def equals(self,value):
    answer=self.all_docs.copy()
    for i in range(len(self.slices)):
      if ((value>>i)&1)==1:
        answer&=self.slices[i]
      else:
        zero_bitmap=self.all_docs-self.slices[i]
        answer&=zero_bitmap
    return answer

  def greater_than(self,value):
    equal=self.all_docs.copy()#initially all documents present here
    greater=BitMap()#initially empty

    for i in reversed(range(len(self.slices))):
      bit=(value>>i)&1
      if bit==1:
      #eg if ith  bit of value is 1; out of all documents only documents whose ith 1 can be tied and later be > value
      #who has ith bit 1 in this case? slice[i] so we intersect from that
        equal&=self.slices[i]
      else:#if i'th bit is zero
        #out of all documents
        #documents with i'th bit 1 automatically become greater
        #remaining all leftover docs must have ith bit 0
        greater|=equal&self.slices[i]
        zero_bitmap=self.all_docs-self.slices[i]
        equal&=zero_bitmap
    return greater

  def greater_than_equals(self,value):
    return self.greater_than(value)|self.equals(value)

  def less_than(self,value):
    return self.all_docs-(self.greater_than(value)|self.equals(value))

  def less_than_equals(self,value):
    return self.less_than(value)|self.equals(value)

  def get_value(self, doc_id): #returns the original value of a document
    value=0
    for i in range(len(self.slices)):
      if(self.slices[i].contains(doc_id)):
        value|=(1<<i)
    return value


In [33]:
bsi = BSI()

for doc in docs:
    bsi.add(doc["id"], doc["age"])


In [34]:
bsi.print_slices()

Slice 0: [0, 1, 3, 4]
Slice 1: [1, 2, 4, 6]
Slice 2: [0, 2, 4, 5]


In [35]:
print(bsi.greater_than(5))

BitMap([2, 4])


In [36]:
print(bsi.greater_than(2))

BitMap([0, 1, 2, 4, 5])


In [37]:
print(bsi.greater_than(7))

BitMap([])


In [38]:
print(bsi.less_than_equals(5))

BitMap([0, 1, 3, 5, 6, 7])
